In [31]:
import pandas as pd
from pathlib import Path

ARTIFACTS_DIR = Path("../artifacts")

train = pd.read_csv(ARTIFACTS_DIR / "train.csv")
validation = pd.read_csv(ARTIFACTS_DIR / "validation.csv")
test = pd.read_csv(ARTIFACTS_DIR / "test.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 22)
Validation: (14471, 22)
Test: (14472, 22)


In [32]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for df in [train, validation, test]:
    for col in date_columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

In [33]:
def create_features(df):
    df = df.copy()

    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / (24 * 60 * 60)

    return df

In [34]:
train_fe = create_features(train)
validation_fe = create_features(validation)
test_fe = create_features(test)

print(train_fe.shape)

(67533, 26)


In [35]:
feature_columns = [
    "customer_state",
    "payment_types",
    "item_count",
    "total_item_price",
    "total_freight_value",
    "unique_products",
    "unique_sellers",
    "payment_records",
    "max_installments",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour",
    "estimated_delivery_days"
]

target_column = "label"

In [36]:
X_train = train_fe[feature_columns].copy()
y_train = train_fe[target_column].copy()

X_val = validation_fe[feature_columns].copy()
y_val = validation_fe[target_column].copy()

X_test = test_fe[feature_columns].copy()
y_test = test_fe[target_column].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (67533, 13)
X_val: (14471, 13)
X_test: (14472, 13)


In [37]:
import sklearn
print(sklearn.__version__)

1.9.0


In [38]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [39]:
categorical_features = [
    "customer_state",
    "payment_types"
]

numerical_features = [
    "item_count",
    "total_item_price",
    "total_freight_value",
    "unique_products",
    "unique_sellers",
    "payment_records",
    "max_installments",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour",
    "estimated_delivery_days"
]

In [40]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [41]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [43]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

In [44]:
print("Processed Train shape:", X_train_processed.shape)
print("Processed Validation shape:", X_val_processed.shape)
print("Processed Test shape:", X_test_processed.shape)

Processed Train shape: (67533, 43)
Processed Validation shape: (14471, 43)
Processed Test shape: (14472, 43)


In [45]:
import joblib

preprocessor_path = ARTIFACTS_DIR / "preprocessor.joblib"

joblib.dump(preprocessor, preprocessor_path)

print("Saved preprocessor to:", preprocessor_path)

Saved preprocessor to: ..\artifacts\preprocessor.joblib


In [46]:
processed_feature_names = preprocessor.get_feature_names_out()

print("Number of final features:", len(processed_feature_names))
print(processed_feature_names)

Number of final features: 43
['num__item_count' 'num__total_item_price' 'num__total_freight_value'
 'num__unique_products' 'num__unique_sellers' 'num__payment_records'
 'num__max_installments' 'num__purchase_month' 'num__purchase_dayofweek'
 'num__purchase_hour' 'num__estimated_delivery_days'
 'cat__customer_state_AC' 'cat__customer_state_AL'
 'cat__customer_state_AM' 'cat__customer_state_AP'
 'cat__customer_state_BA' 'cat__customer_state_CE'
 'cat__customer_state_DF' 'cat__customer_state_ES'
 'cat__customer_state_GO' 'cat__customer_state_MA'
 'cat__customer_state_MG' 'cat__customer_state_MS'
 'cat__customer_state_MT' 'cat__customer_state_PA'
 'cat__customer_state_PB' 'cat__customer_state_PE'
 'cat__customer_state_PI' 'cat__customer_state_PR'
 'cat__customer_state_RJ' 'cat__customer_state_RN'
 'cat__customer_state_RO' 'cat__customer_state_RR'
 'cat__customer_state_RS' 'cat__customer_state_SC'
 'cat__customer_state_SE' 'cat__customer_state_SP'
 'cat__customer_state_TO' 'cat__payment_typ

In [47]:
feature_list_path = ARTIFACTS_DIR / "feature_list.txt"

with open(feature_list_path, "w") as f:
    for feature in processed_feature_names:
        f.write(feature + "\n")

print("Saved feature list to:", feature_list_path)

Saved feature list to: ..\artifacts\feature_list.txt


In [48]:
import numpy as np

np.save(ARTIFACTS_DIR / "X_train_processed.npy", X_train_processed)
np.save(ARTIFACTS_DIR / "X_val_processed.npy", X_val_processed)
np.save(ARTIFACTS_DIR / "X_test_processed.npy", X_test_processed)

y_train.to_csv(ARTIFACTS_DIR / "y_train.csv", index=False)
y_val.to_csv(ARTIFACTS_DIR / "y_val.csv", index=False)
y_test.to_csv(ARTIFACTS_DIR / "y_test.csv", index=False)

print("Processed feature tables and labels saved.")

Processed feature tables and labels saved.


## Feature Engineering Summary

- Created date-based features from purchase time and estimated delivery date.
- Removed identifiers and future information from the feature set.
- Excluded `total_payment_value` because it is almost perfectly correlated with `total_item_price`.
- Numerical missing values were imputed using the median.
- Numerical features were standardized.
- Categorical features were imputed with the most frequent value and one-hot encoded.
- The preprocessing pipeline was fitted on Train only.
- The same fitted pipeline was applied to Validation and Test.
- Final feature count: 43.
- The fitted preprocessor and feature list were saved for reproducibility.